1. Find-S Algorithm
# Aim: To implement the Find-S algorithm using Python to find the most specific hypothesis.
#
# Procedure:
# 1. Import the required libraries.
# 2. Create the training dataset.
# 3. Convert the dataset into a DataFrame.
# 4. Initialize the most specific hypothesis.
# 5. Compare each training example.
# 6. Update the hypothesis for positive examples.
# 7. Display the final most specific hypothesis.


In [ ]:
import pandas as pd

# Training dataset
data = {
    'Sky': ['Sunny', 'Sunny', 'Rainy', 'Sunny'],
    'Temperature': ['Warm', 'Warm', 'Cold', 'Warm'],
    'Humidity': ['Normal', 'High', 'High', 'High'],
    'Wind': ['Strong', 'Strong', 'Strong', 'Strong'],
    'Water': ['Warm', 'Warm', 'Warm', 'Warm'],
    'Forecast': ['Same', 'Same', 'Change', 'Same'],
    'EnjoySport': ['Yes', 'Yes', 'No', 'Yes']
}

df = pd.DataFrame(data)

print("Training Dataset:")
print(df)

# Initialize hypothesis
hypothesis = ['0'] * (len(df.columns) - 1)

# FIND-S Algorithm
for index, row in df.iterrows():
    if row['EnjoySport'] == 'Yes':
        for i in range(len(hypothesis)):
            if hypothesis[i] == '0':
                hypothesis[i] = row.iloc[i]
            elif hypothesis[i] != row.iloc[i]:
                hypothesis[i] = '?'

# Display final hypothesis
print("\nFinal Most Specific Hypothesis:")
print(hypothesis)

Training Dataset:
     Sky Temperature Humidity    Wind Water Forecast EnjoySport
0  Sunny        Warm   Normal  Strong  Warm     Same        Yes
1  Sunny        Warm     High  Strong  Warm     Same        Yes
2  Rainy        Cold     High  Strong  Warm   Change         No
3  Sunny        Warm     High  Strong  Warm     Same        Yes

Final Most Specific Hypothesis:
['Sunny', 'Warm', '?', 'Strong', 'Warm', 'Same']


2. Candidate Elimination Algorithm
# Aim: To implement the Candidate Elimination algorithm using Python to find the specific and general boundaries.
#
# Procedure:
# 1. Import the required libraries.
# 2. Upload the CSV or Excel dataset.
# 3. Read and display the dataset.
# 4. Initialize the specific and general boundaries.
# 5. Process each training example.
# 6. Update the specific and general hypotheses.
# 7. Display the final specific and general boundaries.

In [ ]:
import pandas as pd
from google.colab import files

# Upload CSV/Excel file
uploaded = files.upload()
filename = list(uploaded.keys())[0]

# Read dataset (using read_excel for .xlsx files)
if filename.endswith('.xlsx') or filename.endswith('.xls'):
    df = pd.read_excel(filename)
else:
    df = pd.read_csv(filename)

attributes = list(df.columns[:-1])
target = df.columns[-1]

# Get unique values for each attribute
domains = {
    attr: df[attr].unique().tolist()
    for attr in attributes
}

# Initialize S and G
S = [['Ø'] * len(attributes)]
G = [['?'] * len(attributes)]

# Check whether hypothesis covers an example
def covers(h, x):
    return all(a == '?' or a == b for a, b in zip(h, x))

# Candidate Elimination Algorithm
for _, row in df.iterrows():
    x = row[attributes].tolist()
    label = str(row[target]).strip().lower()

    if label == 'yes':
        # Remove inconsistent hypotheses from G
        G = [g for g in G if covers(g, x)]

        # Generalize S
        new_S = []
        for s in S:
            s_new = s.copy()
            for i in range(len(attributes)):
                if s_new[i] == 'Ø':
                    s_new[i] = x[i]
                elif s_new[i] != x[i] and s_new[i] != '?':
                    s_new[i] = '?'
            new_S.append(s_new)
        S = new_S

    else:
        # Remove hypotheses from S that cover negative example
        S = [s for s in S if not covers(s, x)]

        # Specialize G
        new_G = []
        for g in G:
            if covers(g, x):
                for s in S:
                    for i in range(len(attributes)):
                        if g[i] == '?' and s[i] != '?' and s[i] != 'Ø' and s[i] != x[i]:
                            for value in domains[attributes[i]]:
                                if value != x[i]:
                                    g_new = g.copy()
                                    g_new[i] = value
                                    if covers(g_new, s):
                                        new_G.append(g_new)
            else:
                new_G.append(g)

        # Remove duplicate hypotheses
        G = []
        for h in new_G:
            if h not in G:
                G.append(h)

print("Candidate Elimination Algorithm")
print("\nSpecific Boundary (S):")
for h in S:
    print(h)

print("\nGeneral Boundary (G):")
for h in G:
    print(h)

Saving enjoysport.xlsx to enjoysport (1).xlsx
Candidate Elimination Algorithm

Specific Boundary (S):
['sunny', 'warm', '?', 'strong', '?', '?']

General Boundary (G):
['sunny', '?', '?', '?', '?', '?']
['?', 'warm', '?', '?', '?', '?']


 3. ID3 Decision Tree Algorithm
 # Aim: To implement the ID3 Decision Tree algorithm using Python for classification.
 #
 # Procedure:
 # 1. Import the required libraries.
 # 2. Create the training dataset.
 # 3. Calculate entropy and information gain.
 # 4. Select the attribute with the highest information gain.
 # 5. Build the decision tree recursively.
 # 6. Predict the output for a new sample.
 # 7. Display the decision tree and predicted result.

In [ ]:
import pandas as pd
import math
from collections import Counter
from pprint import pprint

# Create dataset
data = {
    'Outlook': ['sunny', 'sunny', 'overcast', 'rain', 'rain',
                'rain', 'overcast', 'sunny', 'sunny', 'rain',
                'sunny', 'overcast', 'overcast', 'rain'],
    'Temperature': ['hot', 'hot', 'hot', 'mild', 'cool',
                    'cool', 'cool', 'mild', 'cool', 'mild',
                    'mild', 'mild', 'hot', 'mild'],
    'Humidity': ['high', 'high', 'high', 'high', 'normal',
                 'normal', 'normal', 'high', 'normal', 'normal',
                 'normal', 'high', 'normal', 'high'],
    'Wind': ['weak', 'strong', 'weak', 'weak', 'weak',
             'strong', 'strong', 'weak', 'weak', 'weak',
             'strong', 'strong', 'weak', 'strong'],
    'Target': ['no', 'no', 'yes', 'yes', 'yes', 'no', 'yes',
               'no', 'yes', 'yes', 'yes', 'yes', 'yes', 'no']
}

df = pd.DataFrame(data)

print("Training Dataset:")
print(df)

# Calculate entropy
def entropy(target):
    counts = Counter(target)
    total = len(target)
    return -sum(
        (count / total) * math.log2(count / total)
        for count in counts.values()
    )

# Calculate information gain
def information_gain(data, attribute, target):
    total_entropy = entropy(data[target])
    weighted_entropy = 0
    for value in data[attribute].unique():
        subset = data[data[attribute] == value]
        weighted_entropy += (
            len(subset) / len(data)
        ) * entropy(subset[target])
    return total_entropy - weighted_entropy

# Build ID3 decision tree
def id3(data, attributes, target):
    if len(data[target].unique()) == 1:
        return data[target].iloc[0]

    if not attributes:
        return data[target].mode()[0]

    best_attribute = max(
        attributes,
        key=lambda attr: information_gain(data, attr, target)
    )

    tree = {best_attribute: {}}
    for value in data[best_attribute].unique():
        subset = data[data[best_attribute] == value]
        remaining = [
            attr for attr in attributes
            if attr != best_attribute
        ]
        tree[best_attribute][value] = id3(
            subset, remaining, target
        )
    return tree

# Build tree
attributes = list(df.columns[:-1])
tree = id3(df, attributes, 'Target')

print("\nDecision Tree:")
pprint(tree)

# Classify a new sample
def predict(tree, sample):
    if not isinstance(tree, dict):
        return tree

    attribute = next(iter(tree))
    value = sample[attribute]
    if value not in tree[attribute]:
        return "Unknown"
    return predict(tree[attribute][value], sample)

# New sample
new_sample = {
    'Outlook': 'sunny',
    'Temperature': 'cool',
    'Humidity': 'high',
    'Wind': 'strong'
}

result = predict(tree, new_sample)
print("\nNew Sample:", new_sample)
print("Predicted Target:", result)

Training Dataset:
     Outlook Temperature Humidity    Wind Target
0      sunny         hot     high    weak     no
1      sunny         hot     high  strong     no
2   overcast         hot     high    weak    yes
3       rain        mild     high    weak    yes
4       rain        cool   normal    weak    yes
5       rain        cool   normal  strong     no
6   overcast        cool   normal  strong    yes
7      sunny        mild     high    weak     no
8      sunny        cool   normal    weak    yes
9       rain        mild   normal    weak    yes
10     sunny        mild   normal  strong    yes
11  overcast        mild     high  strong    yes
12  overcast         hot   normal    weak    yes
13      rain        mild     high  strong     no

Decision Tree:
{'Outlook': {'overcast': 'yes',
             'rain': {'Wind': {'strong': 'no', 'weak': 'yes'}},
             'sunny': {'Humidity': {'high': 'no', 'normal': 'yes'}}}}

New Sample: {'Outlook': 'sunny', 'Temperature': 'cool', 'Humidit

In [ ]:
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score, confusion_matrix

# Load dataset
iris = load_iris()
X = iris.data
y = iris.target

# Split dataset
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# Create KNN model
model = KNeighborsClassifier(n_neighbors=3)

# Train model
model.fit(X_train, y_train)

# Predict
y_pred = model.predict(X_test)

# Display results
print("K-Nearest Neighbours Classification")
print("Actual Values:", y_test)
print("Predicted Values:", y_pred)

# Calculate accuracy
accuracy = accuracy_score(y_test, y_pred)
print("Accuracy:", accuracy * 100, "%")

# Confusion Matrix
print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred))

# Predict new sample
new_sample = [[5.1, 3.5, 1.4, 0.2]]
prediction = model.predict(new_sample)

print("Predicted Flower:", iris.target_names[prediction[0]])

K-Nearest Neighbours Classification
Actual Values: [1 0 2 1 1 0 1 2 1 1 2 0 0 0 0 1 2 1 1 2 0 2 0 2 2 2 2 2 0 0]
Predicted Values: [1 0 2 1 1 0 1 2 1 1 2 0 0 0 0 1 2 1 1 2 0 2 0 2 2 2 2 2 0 0]
Accuracy: 100.0 %
Confusion Matrix:
[[10  0  0]
 [ 0  9  0]
 [ 0  0 11]]
Predicted Flower: setosa


[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/GowriAtla/ML/blob/main/exp 1,2,3.ipynb)